# Simplified United-MedASR
**Research gap:** Can fine-tuning Whisper-small on a small real medical corpus,
with and without BART semantic post-correction, achieve competitive medical ASR?

**Paper reference:** Banerjee et al., arXiv:2412.00055v1

**Dataset:** `Hani89/medical_asr_recording_dataset` (~8.5h, 6661 samples)

**Pipeline:** Baseline Whisper → Fine-tuned Whisper → Fine-tuned Whisper + BART corrector

## 0. Install dependencies

In [2]:

!pip install -q -U transformers datasets accelerate evaluate jiwer librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.0 MB/s eta 0:00:00


In [3]:
import json
import random
import re
import numpy as np
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import load_dataset, Audio
from transformers import (
    WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor,
    WhisperForConditionalGeneration,
    BartTokenizer, BartForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

Device: cuda
GPU: Tesla T4


## 1. Data Loading and Preparation

In [4]:
raw = load_dataset("Hani89/medical_asr_recording_dataset")
AUDIO_COL = "audio"
TEXT_COL = "sentence"

print(f"Columns: {raw['train'].column_names}")
print(f"Train size: {len(raw['train'])}, Test size: {len(raw['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.49k [00:00<?, ?B/s]

data/train-00000-of-00007-b0c1bd87e30f86(…):   0%|          | 0.00/436M [00:00<?, ?B/s]

data/train-00001-of-00007-bee1da05a7ded4(…):   0%|          | 0.00/450M [00:00<?, ?B/s]

data/train-00002-of-00007-ada4ab56b2c07c(…):   0%|          | 0.00/439M [00:00<?, ?B/s]

...

data/test-00000-of-00002-3c1c46fbaf3b4e6(…):   0%|          | 0.00/389M [00:00<?, ?B/s]

data/test-00001-of-00002-fc73f8b564c10fc(…):   0%|          | 0.00/382M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5328 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1333 [00:00<?, ? examples/s]

Columns: ['audio', 'sentence']
Train size: 5328, Test size: 1333


In [5]:
# Subsample to keep Colab runtime manageable
# 2000 train samples ~= enough for meaningful fine-tuning at 1000 steps
TRAIN_SIZE = 2000
TEST_SIZE  = 400

random.seed(42)
train_ds = raw["train"].shuffle(seed=42).select(range(min(TRAIN_SIZE, len(raw["train"]))))
test_ds  = raw["test"].shuffle(seed=42).select(range(min(TEST_SIZE,  len(raw["test"]))))

print(f"Train: {len(train_ds)}, Test: {len(test_ds)}")
print("Sample:", train_ds[0][TEXT_COL])

Train: 2000, Test: 400
Sample: My head hurts in the back and the pain lasts all day.


## 2. Whisper Fine-tuning on Medical Speech

In [6]:
import librosa

MODEL_NAME = "openai/whisper-small"

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_NAME)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_NAME, language="English", task="transcribe")
processor  = WhisperProcessor.from_pretrained(MODEL_NAME, language="English", task="transcribe")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

In [7]:
def prepare_whisper(batch):
    audio = batch[AUDIO_COL]
    array = np.array(audio["array"], dtype=np.float32).flatten()
    sr = audio["sampling_rate"]
    if sr != 16000:
        array = librosa.resample(array, orig_sr=sr, target_sr=16000)
    batch["input_features"] = feature_extractor(
        array, sampling_rate=16000
    ).input_features[0]
    batch["labels"] = tokenizer(batch[TEXT_COL]).input_ids
    return batch

train_whisper = train_ds.map(prepare_whisper, remove_columns=train_ds.column_names, num_proc=1)
test_whisper  = test_ds.map(prepare_whisper, remove_columns=test_ds.column_names, num_proc=1)
print("Whisper datasets ready")

Map (num_proc=1):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/400 [00:00<?, ? examples/s]

Whisper datasets ready


In [8]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [9]:
wer_metric = evaluate.load("wer")

def compute_wer(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str  = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 4)}

In [ ]:
whisper_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
whisper_model.config.forced_decoder_ids = None
whisper_model.config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-medical",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=100,
    max_steps=1000,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    save_total_limit=2,
)

whisper_trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=training_args,
    train_dataset=train_whisper,
    eval_dataset=test_whisper,
    data_collator=data_collator,
    compute_metrics=compute_wer,

)

whisper_trainer.train()
print("Whisper fine-tuning complete")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss


## 3. BART Fine-tuning for Semantic Correction

In [ ]:
# Build (corrupted, clean) pairs by simulating ASR errors on clean transcripts.
# This mirrors the paper's BART corrector trained on ASR output vs ground truth.

# Common ASR confusion pairs for medical terms
CONFUSIONS = [
    ("hypertension", "high tension"),
    ("diabetes", "die of bees"),
    ("prescription", "pre scription"),
    ("medication", "medic ation"),
    ("diagnosis", "die a nosis"),
    ("symptoms", "sim toms"),
    ("physician", "fis ician"),
    ("patient", "pay shent"),
    ("treatment", "treat meant"),
    ("dosage", "dose age"),
    ("clinical", "klin ical"),
    ("chronic", "kron ik"),
    ("inflammation", "in flam a shun"),
    ("infection", "in feck shun"),
    ("antibiotic", "anti by otic"),
]

def corrupt_text(text: str, p_word_drop=0.05, p_char_swap=0.03) -> str:
    """Simulate ASR errors: medical term confusions + random word drops + char swaps."""
    # Apply known confusion pairs
    for clean, noisy in CONFUSIONS:
        if clean in text.lower():
            text = re.sub(clean, noisy, text, flags=re.IGNORECASE)

    # Random word drop
    words = text.split()
    words = [w for w in words if random.random() > p_word_drop]

    # Random char swap in a word
    result = []
    for w in words:
        if len(w) > 3 and random.random() < p_char_swap:
            i = random.randint(0, len(w) - 2)
            w = w[:i] + w[i+1] + w[i] + w[i+2:]
        result.append(w)

    return " ".join(result)

# Build BART training data from the train split transcripts
clean_texts    = [ex[TEXT_COL] for ex in train_ds]
corrupted_texts = [corrupt_text(t) for t in clean_texts]

# Quick sanity check
for c, n in zip(clean_texts[:3], corrupted_texts[:3]):
    print(f"CLEAN:     {c}")
    print(f"CORRUPTED: {n}")
    print()

In [ ]:
from datasets import Dataset

BART_MODEL_NAME = "facebook/bart-base"
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL_NAME)

bart_train_data = Dataset.from_dict({"input": corrupted_texts, "target": clean_texts})

# Use test transcripts for BART eval
test_clean_texts     = [ex[TEXT_COL] for ex in test_ds]
test_corrupted_texts = [corrupt_text(t) for t in test_clean_texts]
bart_test_data = Dataset.from_dict({"input": test_corrupted_texts, "target": test_clean_texts})

MAX_SRC_LEN = 128
MAX_TGT_LEN = 128

def tokenize_bart(batch):
    # text_target replaces the deprecated as_target_tokenizer() context manager
    model_inputs = bart_tokenizer(
        batch["input"],
        text_target=batch["target"],
        max_length=MAX_SRC_LEN,
        truncation=True,
        padding=False,
    )
    return model_inputs

bart_train_tok = bart_train_data.map(tokenize_bart, batched=True, remove_columns=["input", "target"])
bart_test_tok  = bart_test_data.map(tokenize_bart, batched=True, remove_columns=["input", "target"])
print(f"BART train: {len(bart_train_tok)}, test: {len(bart_test_tok)}")

In [ ]:
bart_model = BartForConditionalGeneration.from_pretrained(BART_MODEL_NAME)

bart_collator = DataCollatorForSeq2Seq(
    tokenizer=bart_tokenizer, model=bart_model, pad_to_multiple_of=8
)

def compute_cer(pred):
    cer_metric = evaluate.load("cer")
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = bart_tokenizer.pad_token_id
    pred_str  = bart_tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = bart_tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    return {"cer": round(cer_metric.compute(predictions=pred_str, references=label_str), 4)}

bart_args = Seq2SeqTrainingArguments(
    output_dir="./bart-medical",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=4e-4,
    warmup_steps=50,
    max_steps=500,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=MAX_TGT_LEN,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

bart_trainer = Seq2SeqTrainer(
    model=bart_model,
    args=bart_args,
    train_dataset=bart_train_tok,
    eval_dataset=bart_test_tok,
    data_collator=bart_collator,
    compute_metrics=compute_cer,
)

bart_trainer.train()
print("BART fine-tuning complete")

## 4. Evaluation: Baseline vs Fine-tuned Whisper vs Whisper+BART

In [ ]:
from transformers import pipeline

def transcribe_with_whisper(model, processor, dataset, batch_size=8):
    """Run Whisper inference on an audio dataset, return list of predicted strings."""
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        max_new_tokens=225,
        device=0 if DEVICE == "cuda" else -1,
    )
    audio_arrays = [ex[AUDIO_COL]["array"] for ex in dataset]
    results = pipe(audio_arrays, batch_size=batch_size)
    return [r["text"].strip() for r in results]

def apply_bart_correction(bart_model, bart_tokenizer, texts, batch_size=16):
    """Run BART seq2seq correction on a list of texts."""
    corrected = []
    bart_model.eval()
    bart_model.to(DEVICE)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = bart_tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SRC_LEN
        ).to(DEVICE)
        with torch.no_grad():
            out = bart_model.generate(**enc, max_new_tokens=MAX_TGT_LEN)
        decoded = bart_tokenizer.batch_decode(out, skip_special_tokens=True)
        corrected.extend(decoded)
    return corrected

def compute_wer_score(predictions, references):
    return round(wer_metric.compute(predictions=predictions, references=references), 4)

references = [ex[TEXT_COL] for ex in test_ds]
print(f"Evaluation on {len(references)} samples")

In [ ]:
# --- Baseline: openai/whisper-small, no fine-tuning ---
print("Running baseline Whisper-small...")
baseline_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
baseline_preds = transcribe_with_whisper(baseline_model, processor, test_ds)
wer_baseline   = compute_wer_score(baseline_preds, references)
print(f"Baseline WER: {wer_baseline}")
del baseline_model
torch.cuda.empty_cache()

In [ ]:
# --- Fine-tuned Whisper only ---
print("Running fine-tuned Whisper-small...")
ft_preds   = transcribe_with_whisper(whisper_model, processor, test_ds)
wer_ft     = compute_wer_score(ft_preds, references)
print(f"Fine-tuned Whisper WER: {wer_ft}")

In [ ]:
# --- Fine-tuned Whisper + BART correction ---
print("Applying BART semantic correction...")
bart_corrected = apply_bart_correction(bart_model, bart_tokenizer, ft_preds)
wer_ft_bart    = compute_wer_score(bart_corrected, references)
print(f"Fine-tuned Whisper + BART WER: {wer_ft_bart}")

## 5. Ablation: With vs Without BART Correction

In [ ]:
# Ablation on a fixed 100-sample subset for clean comparison
ABLATION_N = min(100, len(references))

abl_refs   = references[:ABLATION_N]
abl_preds  = ft_preds[:ABLATION_N]
abl_corrected = bart_corrected[:ABLATION_N]

wer_abl_no_bart  = compute_wer_score(abl_preds, abl_refs)
wer_abl_with_bart = compute_wer_score(abl_corrected, abl_refs)

delta = round(wer_abl_no_bart - wer_abl_with_bart, 4)
print(f"Ablation ({ABLATION_N} samples):")
print(f"  Without BART: {wer_abl_no_bart}")
print(f"  With BART:    {wer_abl_with_bart}")
print(f"  Delta (improvement): {delta}")

In [ ]:
# Qualitative examples
print("--- Qualitative Examples ---\n")
for i in range(5):
    print(f"Reference:    {references[i]}")
    print(f"Whisper-FT:   {ft_preds[i]}")
    print(f"+ BART:       {bart_corrected[i]}")
    print()

## 6. Results Table and Save

In [ ]:
results = {
    "dataset": "Hani89/medical_asr_recording_dataset",
    "whisper_model": MODEL_NAME,
    "bart_model": BART_MODEL_NAME,
    "train_samples": len(train_ds),
    "test_samples": len(test_ds),
    "whisper_max_steps": 1000,
    "bart_max_steps": 500,
    "wer": {
        "baseline_whisper_small": wer_baseline,
        "finetuned_whisper_small": wer_ft,
        "finetuned_whisper_small_plus_bart": wer_ft_bart,
    },
    "ablation": {
        "n_samples": ABLATION_N,
        "without_bart": wer_abl_no_bart,
        "with_bart": wer_abl_with_bart,
        "delta": delta,
    }
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved results.json")
print()
print("=" * 60)
print(f"{'Model':<45} {'WER':>8}")
print("=" * 60)
print(f"{'Baseline whisper-small (no FT)':<45} {wer_baseline:>8.4f}")
print(f"{'Fine-tuned whisper-small':<45} {wer_ft:>8.4f}")
print(f"{'Fine-tuned whisper-small + BART corrector':<45} {wer_ft_bart:>8.4f}")
print("=" * 60)
print(f"\nAblation: BART delta (lower = better) = {delta}")